# The economics prize 1969–2025: laureates, JEL fields, field sequence and gender

Step 2 of the Econ forecast (`../README.md`): which areas of economics the Sveriges Riksbank Prize in Economic Sciences
has gone to, as the basis for the field options of the 2026 forecast (announced Monday 12 October 2026).

**Inputs**

| File | Content |
|---|---|
| `../data/econ_prizes_laureates.csv`, `econ_prizes_by_year.csv` | every prize 1969–2025 from nobelprize.org and the Nobel Prize API (crawl of 9 October 2026; `../data/econ_prizes.md`) |
| `../data/classificationTree.xml` | the JEL classification tree (20 letters, 139 level-2 and 856 level-3 codes) |
| `jel_by_model.csv` | the JEL codes that three models gave every prize work (`classify_jel.py`: claude-opus-5-5, gpt-5.5-2026-04-23, gemini-3.1-pro-preview, from their own knowledge, no web search) |

A **work** is one motivation group of a prize year: 57 prizes, of which 7 were divided between two works, give 64
works. Counts weighted by the **prize share** of a work (the sum of its laureates' portions) add up to 57 prizes.

**Sections:** 1 prizes and laureates · 2 the JEL tree · 3 JEL codes of the prize works and their consensus · 4 the
fields awarded · 5 kind of contribution · 6 field sequence in award order · 7 gender · 8 age of the laureates · 9 the
2026 committee's fields · 10 summary. Figures and tables are written to `results/`.

In [ ]:
import html
import json
import xml.etree.ElementTree as ET
from collections import Counter
from fractions import Fraction
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats
from sklearn.metrics import cohen_kappa_score

ECON = Path('/project/jevans/Dawoon/Nobel Prize/Econ')
DATA, HERE = ECON / 'data', ECON / '02_fields'
OUT = HERE / 'results'
OUT.mkdir(exist_ok=True)
FORECAST_YEAR = 2026
MODELS = {'A': 'claude-opus-5-5', 'O': 'gpt-5.5-2026-04-23', 'G': 'gemini-3.1-pro-preview'}
pd.set_option('display.max_colwidth', 110, 'display.width', 220, 'display.max_rows', 120)

# Chart style: the dataviz reference palette (light), hairline solid grid, thin marks, text in ink tokens
INK, INK2, MUTED, GRID, BASE, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'      # categorical slots 1-4, fixed order
GRAY = '#c3c2b7'                                                            # de-emphasis
BLUES = mpl.colors.LinearSegmentedColormap.from_list(
    'blues', ['#f0efec', '#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'])
mpl.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE, 'figure.dpi': 110,
    'axes.edgecolor': BASE, 'axes.linewidth': 1, 'axes.labelcolor': INK2, 'axes.titlecolor': INK,
    'axes.titlesize': 12, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left', 'axes.labelsize': 10,
    'xtick.color': BASE, 'ytick.color': BASE, 'xtick.labelcolor': INK2, 'ytick.labelcolor': INK2,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'legend.fontsize': 9, 'font.family': 'sans-serif', 'svg.fonttype': 'none', 'lines.linewidth': 2,
    'lines.solid_capstyle': 'round'})


def save(fig, name):
    fig.savefig(OUT / f'{name}.svg', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight', dpi=160)

## 1. Prizes and laureates

Periods: the single prize of 1969 joins the 1970s (`1969–79`), and the current decade runs to 2025 (`2020–25`).

In [ ]:
L = pd.read_csv(DATA / 'econ_prizes_laureates.csv')
Y = pd.read_csv(DATA / 'econ_prizes_by_year.csv')
L['share'] = L['portion'].map(lambda s: float(Fraction(s)))
# A gap in the API (crawl of 9 Oct 2026): Oliver E. Williamson died on 21 May 2020 (UC Berkeley obituary), but his
# laureate record has no death date. Added here only while the API lacks it; the crawled files stay as fetched.
DEATH_ADDED = {'Oliver E. Williamson': '2020-05-21'}
for name, d in DEATH_ADDED.items():
    gap = L['laureate'].eq(name) & L['death_date'].isna()
    L.loc[gap, 'death_date'] = d
    print(f'death date added (missing in the API): {name} {d}' if gap.any() else f'API now has {name}')
L['work_id'] = L['year'].astype(str) + '-' + L['group'].astype(str)
L['alive'] = L['death_date'].isna()
# DATA ERROR in the Nobel Prize API (crawl of 9 Oct 2026): the 2022 prize is dated 2011-10-10, so the API's integer
# age_at_award of Bernanke, Diamond and Dybvig is eleven years too low (57, 57, 56). Where the award date lies outside
# the prize year, the age is recomputed to 10 October of the prize year (68, 68, 67); the crawled file stays as fetched.
_born = pd.to_datetime(L['birth_date'].where(L['birth_date_precision'].eq('day'), L['birth_date'].str[:4] + '-07-01'))
_aw = pd.to_datetime(L['date_awarded'])
_odd = _aw.dt.year != L['year']
L['date_awarded_fixed'] = _aw.where(~_odd, pd.to_datetime(L['year'].astype(str) + '-10-10'))
L.loc[_odd, 'age_at_award'] = np.floor((L['date_awarded_fixed'] - _born).dt.days / 365.2425)[_odd].astype(int)
print('age_at_award corrected (API award date outside the prize year):',
      ', '.join(f"{r.laureate} {r.year}: {int(r.age_at_award)}" for r in L[_odd].itertuples()))
L['woman'] = L['gender'].eq('female')


def period(year):
    if year < 1980:
        return '1969–79'
    if year >= 2020:
        return '2020–25'
    return f'{year // 10 * 10}s'


PERIODS = ['1969–79', '1980s', '1990s', '2000s', '2010s', '2020–25']
L['period'] = pd.Categorical(L['year'].map(period), PERIODS, ordered=True)
assert len(Y) == 57 and len(L) == 99 and L['api_laureate_id'].is_unique
assert Y['year'].tolist() == list(range(1969, 2026))
assert np.allclose(L.groupby('year')['share'].sum(), 1)

P = (L.groupby('year')
      .agg(n_laureates=('laureate', 'size'), n_works=('group', 'nunique'), women=('woman', 'sum'),
           shares=('portion', lambda s: ' + '.join(sorted(s, key=lambda x: -Fraction(x)))))
      .reset_index())
P['period'] = pd.Categorical(P['year'].map(period), PERIODS, ordered=True)
overview = pd.Series({
    'prizes (every year 1969–2025)': len(P),
    'laureates (99 different people; nobody has won twice)': len(L),
    'prize works (motivation groups)': L['work_id'].nunique(),
    'prizes divided between two works': int((P.n_works > 1).sum()),
    'prizes to one laureate': int((P.n_laureates == 1).sum()),
    'prizes to two laureates': int((P.n_laureates == 2).sum()),
    'prizes to three laureates': int((P.n_laureates == 3).sum()),
    'women laureates': int(L.woman.sum()),
    'laureates alive on 12 Oct 2026 (API death dates, one added above)': int(L.alive.sum())})
display(overview.to_frame('count'))

In [ ]:
per = pd.crosstab(P['period'], P['n_laureates']).reindex(PERIODS)
per.columns = [f'{c} laureate' + ('s' if c > 1 else '') for c in per.columns]
per['prizes'] = per.sum(axis=1)
per['laureates'] = P.groupby('period', observed=True)['n_laureates'].sum()
per['laureates per prize'] = (per['laureates'] / per['prizes']).round(2)
per['split between two works'] = P.groupby('period', observed=True)['n_works'].apply(lambda s: int((s > 1).sum()))
display(per)
display(pd.crosstab(P['shares'], P['period']).reindex(columns=PERIODS).assign(total=lambda d: d.sum(axis=1))
          .sort_values('total', ascending=False).rename_axis('shares of the prize'))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
x = np.arange(len(PERIODS))
bottom = np.zeros(len(PERIODS))
for col, color in zip(['1 laureate', '2 laureates', '3 laureates'], [BLUE, ORANGE, AQUA]):
    v = per[col].to_numpy()
    ax.bar(x, v, 0.32, bottom=bottom, color=color, edgecolor=SURFACE, linewidth=2, label=col)
    bottom += v
for xi, tot, r in zip(x, bottom, per['laureates per prize']):
    ax.text(xi, tot + 0.25, f'{r:.1f} per prize', ha='center', va='bottom', fontsize=9, color=INK2)
ax.set_xticks(x, PERIODS)
ax.set_ylabel('prizes')
ax.set_ylim(0, bottom.max() + 3.4)
ax.grid(axis='x', visible=False)
ax.set_title('Prizes by number of laureates: the solo prize has become rare')
ax.legend(loc='upper right', ncol=3)
save(fig, 'laureates_per_prize')
plt.show()

### Affiliation at the prize and country of birth

The API gives each laureate's affiliation at the time of the prize (a laureate can have two). An institution is
counted once per laureate.

In [ ]:
L['countries'] = L['affiliation_countries'].fillna('').map(lambda s: [c for c in s.split(' ; ') if c])
L['us_affiliation'] = L['countries'].map(lambda c: 'USA' in c)
L['born_abroad_us'] = L['us_affiliation'] & L['birth_country_now'].ne('USA')
geo = (L.groupby('period', observed=True)
        .agg(laureates=('laureate', 'size'), us_affiliated=('us_affiliation', 'sum'),
             of_whom_born_outside_us=('born_abroad_us', 'sum')))
geo['share us'] = (geo.us_affiliated / geo.laureates).round(2)
geo.loc['all'] = geo.sum(numeric_only=True)
geo.loc['all', 'share us'] = round(geo.loc['all', 'us_affiliated'] / geo.loc['all', 'laureates'], 2)
display(geo)
print('no affiliation recorded:', ', '.join(L.loc[L.countries.str.len() == 0, 'laureate']))
countries = L.explode('countries')['countries'].value_counts()
display(countries.rename('laureates with an affiliation in the country').to_frame().T)
inst = (L['affiliations_short'].dropna().str.split(' / ')
         .map(lambda xs: list(dict.fromkeys(xs))).explode().value_counts())
display(inst.head(12).rename('laureates').to_frame())

In [ ]:
top = inst.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.barh(np.arange(len(top)), top.values, 0.42, color=BLUE)
for i, v in enumerate(top.values):
    ax.text(v + 0.15, i, str(v), va='center', fontsize=9, color=INK2)
ax.set_yticks(np.arange(len(top)), top.index)
ax.grid(axis='y', visible=False)
ax.set_xlabel('laureates affiliated at the time of the prize')
ax.set_title('Ten most frequent affiliations at the prize')
save(fig, 'affiliations')
plt.show()

## 2. The JEL classification tree

`classificationTree.xml` nests `<classification level=…>` elements, each with a `<code>` and a `<description>`
(the XML writes `&bull;` between the topics of a code). The models were given exactly this list and asked for
level-3 codes.

In [ ]:
def walk(node, parent=None):
    for c in node.findall('classification'):
        code = c.findtext('code').strip()
        yield {'level': int(c.get('level')), 'code': code, 'parent': parent,
               'description': html.unescape(c.findtext('description').strip())}
        yield from walk(c, code)


JEL = pd.DataFrame(walk(ET.parse(DATA / 'classificationTree.xml').getroot()))
assert JEL['code'].is_unique
NAME = JEL.set_index('code')['description'].to_dict()
CODES3 = set(JEL.loc[JEL.level == 3, 'code'])
display(JEL['level'].value_counts().sort_index().rename('codes').to_frame().T)
tree = JEL[JEL.level == 1][['code', 'description']].set_index('code')
tree['level-2 codes'] = JEL[JEL.level == 2].groupby('parent').size()
tree['level-3 codes'] = JEL[JEL.level == 3].assign(l1=lambda d: d.code.str[0]).groupby('l1').size()
display(tree)

# short names of the letters for chart axes (the tables keep the full JEL descriptions)
SHORT = {'A': 'General economics, teaching', 'B': 'History of thought, methodology', 'C': 'Quantitative methods',
         'D': 'Microeconomics', 'E': 'Macro and monetary', 'F': 'International', 'G': 'Financial',
         'H': 'Public', 'I': 'Health, education, welfare', 'J': 'Labor and demographic', 'K': 'Law and economics',
         'L': 'Industrial organization', 'M': 'Business economics', 'N': 'Economic history',
         'O': 'Development, innovation, growth', 'P': 'Political economy, systems',
         'Q': 'Agricultural, resource, environmental', 'R': 'Urban, regional, transport', 'Y': 'Miscellaneous',
         'Z': 'Other special topics'}
assert set(SHORT) == set(tree.index)

## 3. JEL codes of the prize works

`classify_jel.py` sent the 64 works in four batches of consecutive years (the works of a divided prize always in the
same batch) to each of the three models: the year, the laureates with their shares and the official motivation. Each
model returned a **primary** level-3 code for the core of the recognized contribution, up to three **secondary**
codes, the kind of contribution, whether it knew the prize beyond the motivation, and a one-sentence rationale. Code
checked every answer: every work exactly once, every code a level-3 code of the tree (all 12 answers passed the first
time).

**Consensus**, at each level in turn (letter, then level 2 within that letter, then level 3 within that level 2):
the code that at least two of the three primary codes share; if there is none, the code with the highest score (2
per primary code, 1 per model that lists it among its secondary codes); a remaining tie goes to the primary of the
first model in the order A, O, G (flagged `tie`).

In [ ]:
MC = pd.read_csv(HERE / 'jel_by_model.csv', keep_default_na=False)
MC['secondary_list'] = MC['secondary'].map(lambda s: [x for x in s.split(';') if x])
works = (L.groupby('work_id', sort=False)
          .agg(year=('year', 'first'), group=('group', 'first'), period=('period', 'first'),
               laureates=('laureate', '; '.join), n_laureates=('laureate', 'size'), share=('share', 'sum'),
               women=('woman', 'sum'), motivation=('motivation', 'first')))
assert set(MC.work_id) == set(works.index) and MC.groupby('work_id').size().eq(3).all()
assert MC['primary'].isin(CODES3).all() and MC['secondary_list'].map(lambda s: set(s) <= CODES3).all()
print(f"{len(MC)} answers = {MC.work_id.nunique()} works x {MC.letter.nunique()} models; every code is a level-3 "
      f"code of the tree; 'knows the prize': " +
      ', '.join(f"{k} {v:.0%}" for k, v in MC.groupby('letter')['knows_prize'].mean().reindex(list('AOG')).items()))
wide = MC.pivot(index='work_id', columns='letter', values='primary')[list('AOG')].loc[works.index]
display(wide.T)

In [ ]:
rows = []
for depth, label in ((1, 'letter'), (2, 'level 2'), (3, 'level 3')):
    w = wide.apply(lambda col: col.str[:depth])
    distinct = w.nunique(axis=1)
    row = {'level': label, 'all three agree': (distinct == 1).mean(), 'two agree': (distinct == 2).mean(),
           'all differ': (distinct == 3).mean()}
    for a, b in (('A', 'O'), ('A', 'G'), ('O', 'G')):
        row[f'{a}={b}'] = (w[a] == w[b]).mean()
        row[f'kappa {a}-{b}'] = cohen_kappa_score(w[a], w[b])
    rows.append(row)
agreement = pd.DataFrame(rows).set_index('level')
display(agreement.style.format('{:.2f}'))

In [ ]:
ORDER = 'AOG'


def vote(rows, depth, within=''):
    # the depth-character prefix (1 letter, 2 level 2, 3 code) among codes that start with `within`
    prim = {r.letter: r.primary[:depth] for r in rows if r.primary.startswith(within)}
    votes = Counter(prim.values())
    if votes and votes.most_common(1)[0][1] >= 2:
        best = votes.most_common(1)[0][0]
        return best, votes[best], 'majority'
    score = Counter()
    for r in rows:
        own = r.primary[:depth] if r.primary.startswith(within) else None
        if own:
            score[own] += 2
        for s in {x[:depth] for x in r.secondary_list if x.startswith(within)} - {own}:
            score[s] += 1
    top = max(score.values())
    tied = sorted(c for c in score if score[c] == top)
    best = next((prim[m] for m in ORDER if prim.get(m) in tied), tied[0])
    return best, votes.get(best, 0), 'score' if len(tied) == 1 else 'tie'


TYPE_ORDER = ['theory', 'theory_and_empirical', 'empirical', 'methods']
out = []
for wid, g in MC.groupby('work_id', sort=False):
    rs = list(g.itertuples())
    l1, a1, r1 = vote(rs, 1)
    l2, a2, r2 = vote(rs, 2, l1)
    l3, a3, r3 = vote(rs, 3, l2)
    types = Counter(g['contribution_type'])
    t, n = types.most_common(1)[0]
    out.append({'work_id': wid, 'jel': l3, 'jel_l2': l2, 'jel_l1': l1,
                'agree_l1': a1, 'agree_l2': a2, 'agree_l3': a3, 'rule': '/'.join(dict.fromkeys([r1, r2, r3])),
                'contribution': t if n >= 2 else 'no majority',
                **{f'code_{r.letter}': r.primary + (f" ({', '.join(r.secondary_list)})" if r.secondary_list else '')
                   for r in rs}})
C3 = works.join(pd.DataFrame(out).set_index('work_id'))
C3['field'] = C3['jel_l1'].map(NAME)
C3['field_l2'] = C3['jel_l2'].map(NAME)
C3['code_name'] = C3['jel'].map(NAME)
display(C3['rule'].value_counts().rename('works').to_frame().T)
display(C3[['year', 'laureates', 'jel', 'code_name', 'field_l2', 'agree_l1', 'agree_l2', 'agree_l3', 'contribution']])

Where the three models do not share a letter, the codes show how the work straddles fields. The full rationales are
in `jel_by_model.csv`.

In [ ]:
split = C3[C3.agree_l1 < 3]
display(split[['laureates', 'motivation', 'code_A', 'code_O', 'code_G', 'jel', 'rule']])

## 4. The fields the prize has gone to

Three counts per JEL letter: **works** (consensus code), **prizes** (works weighted by their prize share, adding up to
57), **soft prizes** (each model's primary code gets a third of the work's share; no consensus step), and **any
mention** (works in which at least one model gives the letter as a primary or secondary code).

In [ ]:
share = C3.groupby('jel_l1')['share']
soft = (MC.assign(l1=MC.primary.str[0]).join(works['share'], on='work_id')
          .assign(w=lambda d: d.share / 3).groupby('l1')['w'].sum())
mention = Counter(l for wid, g in MC.groupby('work_id')
                  for l in {c[0] for c in g.primary} | {c[0] for s in g.secondary_list for c in s})
F1 = tree[['description']].copy()
F1['works'] = C3.groupby('jel_l1').size()
F1['prizes'] = share.sum()
F1['soft prizes'] = soft
F1['any mention'] = pd.Series(mention)
F1['years'] = C3.groupby('jel_l1')['year'].apply(lambda s: ', '.join(map(str, sorted(set(s)))))
F1 = F1.fillna({'works': 0, 'prizes': 0, 'soft prizes': 0, 'any mention': 0, 'years': ''})
F1 = F1.astype({'works': int, 'any mention': int}).sort_values(['prizes', 'soft prizes'], ascending=False)
display(F1.style.format({'prizes': '{:.1f}', 'soft prizes': '{:.2f}'}))
never = F1[F1.works == 0]
print('letters never the consensus field:', ', '.join(f'{c} ({d})' for c, d in never['description'].items()))

In [ ]:
F1a = F1[F1.works > 0].iloc[::-1]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
y = np.arange(len(F1a))
ax.barh(y, F1a['prizes'], 0.42, color=BLUE)
for yi, (p, n) in enumerate(zip(F1a['prizes'], F1a['works'])):
    ax.text(p + 0.2, yi, f'{p:.1f}  ({n} works)', va='center', fontsize=9, color=INK2)
ax.set_yticks(y, [f'{c}  {SHORT[c]}' for c in F1a.index])
ax.grid(axis='y', visible=False)
ax.set_xlim(0, F1a['prizes'].max() + 4)
ax.set_xlabel('prizes (works weighted by their share of the prize)')
ax.set_title('Prizes by JEL letter of the recognized work, 1969–2025')
save(fig, 'fields_jel_letter')
plt.show()

In [ ]:
F2 = (C3.groupby('jel_l2')
        .agg(works=('year', 'size'), prizes=('share', 'sum'), years=('year', lambda s: ', '.join(map(str, s))),
             laureates=('laureates', ' | '.join)))
F2.insert(0, 'letter', F2.index.str[0])
F2.insert(1, 'field', F2.index.map(NAME))
F2 = F2.sort_values(['prizes', 'works'], ascending=False)
print(f'{len(F2)} level-2 fields and {C3.jel.nunique()} level-3 codes are a consensus field of at least one work')
display(F2[['letter', 'field', 'works', 'prizes', 'years']].style.format({'prizes': '{:.1f}'}))

In [ ]:
heat = C3.pivot_table(index='jel_l1', columns='period', values='share', aggfunc='sum', fill_value=0, observed=False)
heat = heat.reindex(columns=PERIODS, fill_value=0).loc[F1[F1.works > 0].index]
fig, ax = plt.subplots(figsize=(7.5, 4.4))
im = ax.imshow(heat.to_numpy(), cmap=BLUES, aspect='auto', vmin=0, vmax=heat.to_numpy().max())
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.iat[i, j]
        if v:
            ax.text(j, i, f'{v:g}', ha='center', va='center', fontsize=8.5,
                    color='white' if v >= 0.55 * heat.to_numpy().max() else INK)
ax.set_xticks(range(len(PERIODS)), PERIODS)
ax.set_yticks(range(len(heat)), [f'{c}  {SHORT[c]}' for c in heat.index])
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title('Prizes by JEL letter and period (share-weighted)')
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.outline.set_visible(False)
cb.ax.tick_params(color=BASE, labelcolor=INK2, labelsize=8)
save(fig, 'fields_by_period')
plt.show()
display(heat.assign(total=heat.sum(axis=1)))

### The awarded JEL fields, level 2 within letter

The list below is the candidate option set for a field question: every level-2 JEL field that is the consensus field
of at least one prize work, with the years and laureates. It is written to `results/awarded_jel_fields.csv` and
`results/awarded_jel_fields.md`.

In [ ]:
aw = (C3.reset_index()
        .assign(item=lambda d: d.year.astype(str) + ' ' + d.laureates.str.replace(r' \(.*?\)', '', regex=True)
                + ' [' + d.jel + ']')
        .groupby('jel_l2')
        .agg(works=('work_id', 'size'), prizes=('share', 'sum'), first=('year', 'min'), last=('year', 'max'),
             codes=('jel', lambda s: ', '.join(sorted(set(s)))), awards=('item', ' | '.join)))
aw.insert(0, 'letter', aw.index.str[0])
aw.insert(1, 'letter_name', aw['letter'].map(NAME))
aw.insert(2, 'field', aw.index.map(NAME))
aw['years_since_last'] = FORECAST_YEAR - aw['last']
aw = aw.sort_values(['letter', 'jel_l2'])
aw.to_csv(OUT / 'awarded_jel_fields.csv', index_label='jel_l2')

lines = ['# JEL fields of the economics prize, 1969-2025', '',
         'Consensus JEL codes of the 64 prize works (57 prizes) from three models (claude-opus-5-5, '
         'gpt-5.5-2026-04-23, gemini-3.1-pro-preview), built in `prize_history.ipynb`. Prizes = works weighted by '
         'their share of the prize.', '']
for l1, g in aw.groupby('letter'):
    lines += [f'## {l1} {NAME[l1]} ({g.works.sum()} works, {g.prizes.sum():g} prizes)', '',
              '| JEL | Field | Works | Prizes | Awards (year, laureates, code) |', '|---|---|---|---|---|']
    lines += [f"| {k} | {r.field} | {r.works} | {r.prizes:g} | {r.awards} |" for k, r in g.iterrows()]
    lines.append('')
lines += ['## Never the consensus field', '', ', '.join(f'{c} {d}' for c, d in never['description'].items()), '']
(OUT / 'awarded_jel_fields.md').write_text('\n'.join(lines), encoding='utf-8')
display(aw[['letter', 'field', 'works', 'prizes', 'first', 'last', 'awards']].style.format({'prizes': '{:g}'}))

## 5. Kind of contribution

Each model labelled the recognized work as theory, theory and empirical, empirical, or methods; the consensus is the
label that at least two models give.

In [ ]:
ct = pd.crosstab(C3['period'], C3['contribution']).reindex(PERIODS)
ct = ct[[c for c in TYPE_ORDER + ['no majority'] if c in ct.columns]]
display(ct.assign(works=ct.sum(axis=1)))
print('model agreement on the kind of contribution: all three',
      f"{(MC.groupby('work_id').contribution_type.nunique() == 1).mean():.0%} of works")

In [ ]:
shares = ct.div(ct.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
x = np.arange(len(PERIODS))
bottom = np.zeros(len(PERIODS))
labels = {'theory': 'theory', 'theory_and_empirical': 'theory and empirical', 'empirical': 'empirical',
          'methods': 'methods', 'no majority': 'no majority'}
for col, color in zip(shares.columns, [BLUE, ORANGE, AQUA, YELLOW, GRAY]):
    v = shares[col].to_numpy()
    ax.bar(x, v, 0.32, bottom=bottom, color=color, edgecolor=SURFACE, linewidth=2, label=labels[col])
    for xi, b, vi in zip(x, bottom, v):
        if vi >= 0.12:
            ax.text(xi + 0.2, b + vi / 2, f'{vi:.0%}', va='center', fontsize=8, color=INK2)
    bottom += v
ax.set_xticks(x, PERIODS)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
ax.set_ylim(0, 1)
ax.grid(axis='x', visible=False)
ax.set_title('Kind of recognized contribution, share of works per period')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
save(fig, 'contribution_kind')
plt.show()

## 6. Field sequence in award order

One entry per year: the set of consensus letters of the year's works (one, or two for a divided prize). The
**repeat** tests compare the observed number of years that share a letter with the previous year (or with any of the
previous two or three years) against 20,000 random orderings of the same 57 yearly sets: a low count means the
committee moves on from a field more often than chance would.

In [ ]:
seq = C3.groupby('year')['jel_l1'].apply(lambda s: frozenset(s))
seq2 = C3.groupby('year')['jel_l2'].apply(lambda s: frozenset(s))
rows = []
for decade in range(1960, 2030, 10):
    ys = [y for y in seq.index if decade <= y < decade + 10]
    rows.append({'decade': f'{decade}s', **{str(y % 10): '+'.join(sorted(seq[y])) for y in ys}})
display(pd.DataFrame(rows).set_index('decade').reindex(columns=[str(i) for i in range(10)]).fillna(''))


def repeats(sets, window=1):
    return sum(bool(sets[i] & frozenset().union(*sets[max(0, i - window):i])) for i in range(1, len(sets)))


rng = np.random.default_rng(2026)
rows = []
for label, s in (('letter', list(seq)), ('level 2', list(seq2))):
    for window in (1, 2, 3):
        obs = repeats(s, window)
        null = np.array([repeats([s[i] for i in rng.permutation(len(s))], window) for _ in range(20000)])
        rows.append({'level': label, 'window (years)': window, 'observed repeats': obs,
                     'expected (random order)': null.mean(), 'P(random <= observed)': (null <= obs).mean(),
                     'P(random >= observed)': (null >= obs).mean()})
rep = pd.DataFrame(rows)
display(rep.style.format({'expected (random order)': '{:.1f}', 'P(random <= observed)': '{:.3f}',
                          'P(random >= observed)': '{:.3f}'}))
same = [(y, '+'.join(sorted(seq[y] & seq[y - 1]))) for y in seq.index[1:] if seq[y] & seq[y - 1]]
print('years sharing a letter with the year before:', ', '.join(f'{y} ({l})' for y, l in same))

In [ ]:
order = list(F1[F1.works > 0].index)
ypos = {c: i for i, c in enumerate(order[::-1])}
fig, ax = plt.subplots(figsize=(9, 4.4))
lead = C3.sort_values(['year', 'group']).groupby('year').first()
ax.plot(lead.index, [ypos[c] for c in lead['jel_l1']], color=GRID, linewidth=1, zorder=1)
recent = C3['year'] >= 2016
for mask, color, label in ((~recent, BLUE, '1969–2015'), (recent, ORANGE, '2016–2025 (last ten prizes)')):
    d = C3[mask]
    ax.scatter(d['year'], d['jel_l1'].map(ypos), s=d['share'] * 70, color=color, edgecolor=SURFACE, linewidth=1.5,
               zorder=2, label=label)
ax.set_yticks(list(ypos.values()), [f'{c}  {SHORT[c]}' for c in ypos])
ax.set_xlim(1966, 2028)
ax.grid(axis='y', color=GRID)
ax.grid(axis='x', visible=False)
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), markerscale=0.8)
ax.set_title('JEL letter of each prize work in award order; a half-size dot is half a prize')
save(fig, 'field_sequence')
plt.show()

In [ ]:
idx = {c: i for i, c in enumerate(order)}
T = np.zeros((len(order), len(order)))
yrs = list(seq.index)
for a, b in zip(yrs[:-1], yrs[1:]):
    for x in seq[a]:
        for y in seq[b]:
            T[idx[x], idx[y]] += 1 / (len(seq[a]) * len(seq[b]))
T = pd.DataFrame(T, index=order, columns=order)
print('from the field of year t (rows) to the field of year t+1 (columns); a divided year splits its weight')
display(T.round(2).style.format('{:g}').background_gradient(cmap=BLUES, axis=None))

### Gaps between awards to the same field

For each letter (and each level-2 field): the award years, the gaps between them, and the years since the last award
as of 2026. A field whose time since its last prize exceeds its median gap is past its usual return interval, which is
a description of the record, not a forecast.

In [ ]:
def gaps(col, min_years=2):
    rows = []
    for code, g in C3.groupby(col):
        ys = sorted(set(g['year']))
        d = np.diff(ys)
        rows.append({col: code, 'field': NAME[code], 'award years': len(ys), 'works': len(g),
                     'prizes': g['share'].sum(), 'first': ys[0], 'last': ys[-1],
                     'median gap': float(np.median(d)) if len(d) else np.nan,
                     'longest gap': int(d.max()) if len(d) else np.nan,
                     'years since last': FORECAST_YEAR - ys[-1], 'years': ', '.join(map(str, ys))})
    t = pd.DataFrame(rows).set_index(col)
    t['since / median gap'] = (t['years since last'] / t['median gap']).round(2)
    return t[t['award years'] >= min_years].sort_values('since / median gap', ascending=False)


G1 = gaps('jel_l1')
mentions = MC.assign(l=MC.apply(lambda r: sorted({r.primary[0]} | {x[0] for x in r.secondary_list}), axis=1))
G1.insert(6, 'last any mention', mentions.explode('l').groupby('l')['year'].max().reindex(G1.index))
G2 = gaps('jel_l2')
G1.to_csv(OUT / 'field_gaps_letter.csv')
G2.to_csv(OUT / 'field_gaps_level2.csv')
display(G1.drop(columns=['years']).style.format({'prizes': '{:.1f}', 'median gap': '{:g}', 'longest gap': '{:g}'}))
display(G2.drop(columns=['years']).style.format({'prizes': '{:.1f}', 'median gap': '{:g}', 'longest gap': '{:g}'}))

In [ ]:
g = G1[G1['award years'] >= 3].sort_values('years since last')
fig, ax = plt.subplots(figsize=(7.5, 3.8))
y = np.arange(len(g))
ax.hlines(y, g[['median gap', 'years since last']].min(axis=1), g[['median gap', 'years since last']].max(axis=1),
          color=GRID, linewidth=2)
ax.scatter(g['median gap'], y, s=60, color='#86b6ef', edgecolor=SURFACE, linewidth=2, zorder=3, label='median gap')
ax.scatter(g['years since last'], y, s=60, color='#184f95', edgecolor=SURFACE, linewidth=2, zorder=3,
           label=f'years since the last award (to {FORECAST_YEAR})')
ax.set_yticks(y, [f'{c}  {SHORT[c]}' for c in g.index])
ax.grid(axis='y', visible=False)
ax.set_xlabel('years')
ax.set_title('Time since the last award against the usual gap, JEL letters with 3+ award years')
ax.legend(loc='lower right')
save(fig, 'field_gaps')
plt.show()

### Theory and empirical work in sequence

A runs test (Wald–Wolfowitz) on the yearly sequence "theory" versus "not theory" (the consensus kind of the year's
first work): fewer runs than expected would mean streaks, more runs would mean alternation.

In [ ]:
kind = C3.sort_values(['year', 'group']).groupby('year')['contribution'].first()
b = (kind == 'theory').to_numpy()
n1, n2 = b.sum(), (~b).sum()
runs = 1 + int((b[1:] != b[:-1]).sum())
mu = 2 * n1 * n2 / (n1 + n2) + 1
var = 2 * n1 * n2 * (2 * n1 * n2 - n1 - n2) / ((n1 + n2) ** 2 * (n1 + n2 - 1))
z = (runs - mu) / np.sqrt(var)
print(f'theory years {n1}, other years {n2}; runs {runs}, expected {mu:.1f}, z = {z:.2f}, '
      f'two-sided p = {2 * stats.norm.sf(abs(z)):.2f}')
print('last ten years:', ', '.join(f'{y} {k}' for y, k in kind.tail(10).items()))

## 7. Gender

In [ ]:
W = L[L.woman].merge(C3[['jel', 'field', 'field_l2', 'contribution']], left_on='work_id', right_index=True)
display(W[['year', 'laureate', 'portion', 'age_at_award', 'affiliations_short', 'motivation', 'jel', 'field_l2']]
        .set_index('year'))
gp = L.groupby('period', observed=True).agg(laureates=('laureate', 'size'), women=('woman', 'sum'))
gp['share women'] = (gp.women / gp.laureates).round(3)
gp.loc['all'] = [len(L), L.woman.sum(), round(L.woman.mean(), 3)]
display(gp)
first = L.loc[L.woman, 'year'].min()
since = L[L.year >= first]
print(f'first woman {first} (after {first - 1969} prizes); since then {since.woman.sum()} of {len(since)} '
      f'laureates ({since.woman.mean():.1%}) and {since.groupby("year").woman.any().sum()} of '
      f'{since.year.nunique()} prizes include a woman')
print(f'prize share held by women: {L.loc[L.woman, "share"].sum():.2f} of 57 prizes; mean age at award: women '
      f'{L.loc[L.woman, "age_at_award"].mean():.1f}, men {L.loc[~L.woman, "age_at_award"].mean():.1f}')
print('women among living laureates:', int((L.woman & L.alive).sum()), 'of', int(L.alive.sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))
men, women = L[~L.woman], L[L.woman]
ax.scatter(men['year'], men['age_at_award'], s=22, color=GRAY, edgecolor=SURFACE, linewidth=1, label='men')
ax.scatter(women['year'], women['age_at_award'], s=70, color=ORANGE, edgecolor=SURFACE, linewidth=2, zorder=3,
           label='women')
for _, r in women.iterrows():
    ax.annotate(f"{r['laureate']} ({r['year']})", (r['year'], r['age_at_award']), xytext=(-8, 9),
                textcoords='offset points', ha='right', fontsize=9, color=INK)
ax.set_xlim(1967, 2028)
ax.set_ylabel('age at the award')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
ax.set_title('The three women laureates among 99, by year and age at the award')
save(fig, 'gender')
plt.show()

## 8. Age of the laureates

Age at the award is the age on the award date (the API's `date_awarded`, except for the 2022 prize, which the API dates 2011-10-10: corrected to the prize year in section 1); four laureates (Romer, Card,
Johnson, Robinson) have only a birth year, so their ages are the year difference. Current ages and ages at death are
counted to Monday 12 October 2026, the day of the 2026 announcement (a birth year alone is dated 1 July).

In [ ]:
a = L['age_at_award']
q = a.quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95])
dist = pd.Series({'laureates': len(a), 'mean': a.mean(), 'sd': a.std(), 'min': a.min(),
                  **{f'p{int(round(k * 100))}': v for k, v in q.items()}, 'max': a.max()})
display(dist.round(1).to_frame('age at the award').T)
bands = pd.cut(a, [40, 50, 55, 60, 65, 70, 75, 80, 85, 95], right=False)
display(bands.value_counts().sort_index().rename('laureates').to_frame().T)
print(f'middle 80% of laureates: {q[0.10]:.0f} to {q[0.90]:.0f}; middle half: {q[0.25]:.0f} to {q[0.75]:.0f}; '
      f'under 55: {(a < 55).sum()}; 80 or older: {(a >= 80).sum()}; 85 or older: {(a >= 85).sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.2))
edges = np.arange(45, 95, 5)
counts, _ = np.histogram(a, bins=edges)
ax.bar(edges[:-1] + 2.5, counts, 4.6, color=BLUE, edgecolor=SURFACE, linewidth=2)
for x, c in zip(edges[:-1] + 2.5, counts):
    ax.text(x, c + 0.4, str(c), ha='center', fontsize=8.5, color=INK2)
ax.axvline(a.median(), color=INK2, linewidth=1)
ax.text(a.median() + 0.6, counts.max() + 2.2, f'median {a.median():.0f}', fontsize=9, color=INK2)
ax.set_xticks(edges, [str(e) for e in edges])
ax.set_ylim(0, counts.max() + 4)
ax.grid(axis='x', visible=False)
ax.set_xlabel('age at the award (5-year bins: 45–49, 50–54, …)')
ax.set_ylabel('laureates')
ax.set_title('Age at the award: most laureates are in their sixties')
save(fig, 'age_distribution')
plt.show()

### Over time

In [ ]:
display(L.groupby('period', observed=True)['age_at_award'].agg(['count', 'mean', 'median', 'min', 'max']).round(1))
fit = stats.linregress(L['year'], L['age_at_award'])
rho = stats.spearmanr(L['year'], L['age_at_award'])
print(f'linear trend: {fit.slope * 10:+.2f} years of age per decade of award (p = {fit.pvalue:.2f}); '
      f'Spearman rho {rho.statistic:+.2f} (p = {rho.pvalue:.2f})')
cols = ['year', 'laureate', 'age_at_award', 'motivation']
display(pd.concat([L.nsmallest(5, 'age_at_award')[cols].assign(rank='youngest'),
                   L.nlargest(5, 'age_at_award')[cols].assign(rank='oldest')]).set_index('rank'))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.scatter(L['year'], L['age_at_award'], s=22, color=BLUE, alpha=0.55, edgecolor=SURFACE, linewidth=1)
ys = np.arange(L.year.min() + 5, L.year.max() - 4)
med = pd.Series([L.loc[L.year.between(y - 5, y + 5), 'age_at_award'].median() for y in ys], index=ys)
ax.plot(med.index, med.values, color=INK2, linewidth=2)
ax.text(med.index[-1] + 0.8, med.values[-1], 'median of the\nlaureates within\n±5 years', fontsize=8.5, color=INK2,
        va='center')
ax.set_xlim(1967, 2034)
ax.set_ylabel('age at the award')
ax.set_title('Age of laureates at the award, one dot per laureate')
save(fig, 'age_at_award')
plt.show()

### By field, kind of contribution and number of laureates

The field and the kind of contribution are those of the laureate's prize work (consensus of section 3). The
Kruskal–Wallis test asks whether the age distributions differ between groups (letters with at least three
laureates).

In [ ]:
LA = L.merge(C3[['jel_l1', 'jel_l2', 'contribution']], left_on='work_id', right_index=True)
LA['n_laureates'] = LA['year'].map(P.set_index('year')['n_laureates'])
byf = LA.groupby('jel_l1')['age_at_award'].agg(['count', 'median', 'mean', 'min', 'max']).round(1)
byf.insert(0, 'field', byf.index.map(SHORT))
byf = byf.sort_values('median')
display(byf)
groups = [g.to_numpy() for _, g in LA.groupby('jel_l1')['age_at_award'] if len(g) >= 3]
kw = stats.kruskal(*groups)
print(f'Kruskal-Wallis across {len(groups)} letters: H = {kw.statistic:.1f}, p = {kw.pvalue:.2f}')
byk = LA.groupby('contribution')['age_at_award'].agg(['count', 'median', 'mean', 'min', 'max']).round(1)
display(byk.reindex([t for t in TYPE_ORDER if t in byk.index]))
kw2 = stats.kruskal(*[g.to_numpy() for _, g in LA.groupby('contribution')['age_at_award']])
print(f'Kruskal-Wallis across kinds of contribution: H = {kw2.statistic:.1f}, p = {kw2.pvalue:.2f}')
byn = LA.groupby('n_laureates')['age_at_award'].agg(['count', 'median', 'mean', 'min', 'max']).round(1)
display(byn.rename_axis('laureates in the prize'))

In [ ]:
ordr = list(byf.index)
fig, ax = plt.subplots(figsize=(7.5, 3.9))
for i, c in enumerate(ordr):
    v = LA.loc[LA.jel_l1 == c, 'age_at_award'].to_numpy()
    jitter = np.random.default_rng(i).uniform(-0.12, 0.12, len(v))
    ax.scatter(v, i + jitter, s=24, color=BLUE, alpha=0.6, edgecolor=SURFACE, linewidth=1, zorder=2)
    ax.plot([np.median(v)] * 2, [i - 0.3, i + 0.3], color=INK, linewidth=2, zorder=3)
ax.set_yticks(range(len(ordr)), [f'{c}  {SHORT[c]} (n={int(byf.loc[c, "count"])})' for c in ordr])
ax.grid(axis='y', visible=False)
ax.set_xlabel('age at the award (bar = median)')
ax.set_title('Age at the award by JEL letter of the prize work')
save(fig, 'age_by_field')
plt.show()

In [ ]:
spread = (L.groupby('year')['age_at_award'].agg(['size', 'min', 'max'])
           .query('size > 1').assign(gap=lambda d: d['max'] - d['min']))
print(f'shared prizes: {len(spread)}; age gap between the oldest and youngest co-laureate: median '
      f'{spread.gap.median():.0f} years, mean {spread.gap.mean():.1f}, largest {spread.gap.max()}')
big = spread.nlargest(5, 'gap').join(L.groupby('year')['laureate'].agg('; '.join))
display(big[['min', 'max', 'gap', 'laureate']])

### Living laureates and lifespans

The prize is not awarded posthumously, so the age profile of the living laureates and the ages at death bound the
age of a plausible candidate.

In [ ]:
ANNOUNCE = pd.Timestamp('2026-10-12')
born = pd.to_datetime(L['birth_date'].where(L['birth_date'].str.len() == 10, L['birth_date'] + '-07-01'))
died = pd.to_datetime(L['death_date'])
awarded = L['date_awarded_fixed']          # API award date corrected in section 1
L['age_now'] = ((ANNOUNCE - born).dt.days / 365.2425).where(L.alive).round(1)
L['age_at_death'] = ((died - born).dt.days / 365.2425).round(1)
L['years_after_prize'] = ((died.fillna(ANNOUNCE) - awarded).dt.days / 365.2425).round(1)
liv, dead = L[L.alive], L[~L.alive]
print(f'living laureates: {len(liv)}; current age median {liv.age_now.median():.1f}, '
      f'youngest {liv.age_now.min():.0f}, oldest {liv.age_now.max():.0f}; aged 80+ now: {(liv.age_now >= 80).sum()}')
print(f'deceased laureates: {len(dead)}; age at death median {dead.age_at_death.median():.1f} '
      f'(range {dead.age_at_death.min():.0f}–{dead.age_at_death.max():.0f}); years lived after the prize: median '
      f'{dead.years_after_prize.median():.1f}')
display(pd.cut(liv.age_now, [40, 60, 70, 80, 90, 100], right=False).value_counts().sort_index()
          .rename('living laureates by current age').to_frame().T)
cols = ['year', 'laureate', 'age_at_award', 'age_now']
display(pd.concat([liv.nlargest(5, 'age_now')[cols].assign(rank='oldest living'),
                   liv.nsmallest(5, 'age_now')[cols].assign(rank='youngest living')]).set_index('rank'))
soon = dead[dead.years_after_prize < 3][['year', 'laureate', 'age_at_award', 'death_date', 'years_after_prize']]
print('died within three years of the prize:')
display(soon.set_index('year'))

## 9. The 2026 committee's fields against the prize fields

The step-1 member profiles (`../01_committee/profiles/<slug>.json`) list each member's fields with JEL codes. Below,
a member covers a letter when one of their merged fields carries a code in it; the table sets these letters beside
the prize record. (Skipped if the profiles are not merged yet.)

In [ ]:
prof_dir = ECON / '01_committee' / 'profiles'
recs = [json.loads(p.read_text()) for p in sorted(prof_dir.glob('*.json'))] if prof_dir.exists() else []
recs = [r for r in recs if r.get('valid')]
if not recs:
    print('no merged profiles yet')
else:
    cover = {}
    for r in recs:
        letters = Counter()
        for f in r['profile']['fields']:
            for c in {x.strip().upper()[:1] for x in f['jel_codes'] if x.strip()[:1].isalpha()}:
                letters[c] += 1
        cover[f"{r['name']} ({r['role']})"] = letters
    CM = pd.DataFrame(cover).T.fillna(0).astype(int)
    CM = CM[[c for c in tree.index if c in CM.columns]]
    display(CM.style.format(lambda v: '●' if v else '').set_caption(
        'letters in which a member has at least one merged field'))
    cmp_ = pd.DataFrame({'members covering': (CM > 0).sum(), 'prizes 1969-2025': F1['prizes']}).fillna(0)
    cmp_['field'] = cmp_.index.map(NAME)
    display(cmp_[cmp_.sum(axis=1, numeric_only=True) > 0].sort_values('prizes 1969-2025', ascending=False))

## 10. Summary

(The numbers are those of the cells above; the table below recomputes the main ones.)

**Prizes and laureates.** 57 prizes to 99 people (nobody twice) for 64 works. The prize is shared more and more:
1.0 laureates per prize in the 1980s, 2.0 in the 2000s and 2010s, 2.5 in 2020–25 (one solo prize since 2020,
Goldin 2023; the split 1/2 + 1/4 + 1/4 only in 2021 and 2025). 82% of laureates had a US affiliation at the prize
(at least 90% in every period since 2000); Chicago (15), MIT (10) and Harvard (8) lead.

**Fields (JEL).** The three models agree on the letter for 58 of the 64 works (pairwise kappa 0.90–0.96), on the
level-2 field for 80% and on the level-3 code for 61%; every work has a two-model majority at the letter and level-2
levels. By prize share: D microeconomics 16.5, C quantitative methods 13, E macroeconomics 8, O development and growth
6, G finance 4, F international 3, J labour 2.5, L industrial organization 2, N economic history 1.5, Q environment
0.5. Ten letters are never the main field (A, B, H, I, K, M, P, R, Y, Z), although public economics (H), health,
education and welfare (I), law (K) and political economy (P) appear among the secondary codes. 35 level-2 fields have
won; the most frequent are O4 growth (5 works), D8 information (4) and C2 single-equation econometrics (4).

**Shifts.** E was the main field eight times between 1974 and 2006 and not since (20 years against a median gap of
3), although macroeconomics is a secondary code of 2011 (Sargent, Sims) and 2022 (Bernanke, Diamond, Dybvig) for all
three models. The last ten prizes: 2016 D, 2017 D, 2018 O + Q, 2019 O, 2020 D, 2021 C + J, 2022 G, 2023 J, 2024 O,
2025 N + O. Growth and development, labour and empirical work have gained; general equilibrium (D5, last 1988) and
mathematical modelling (C5, C6, last 1980) have not returned.

**Sequence.** No sign that the committee avoids, or favours, the field of the year before: 10 years share a letter
with the previous year against 10.2 expected in a random order (P = 0.56; the same for two- and three-year windows).
At level 2 a field followed itself only once (O4, 2024 → 2025; 1.5 expected). Theory and other work do not alternate
more than chance (runs test p = 0.11), but empirical work rose to 3 of 8 works in 2020–25.

**Gender.** 3 women among the 99 laureates: Ostrom (2009, half a prize), Duflo (2019, a third) and Goldin (2023,
alone); since the first, 3 of 37 laureates (8%).

**Age.** Median age at the award 67 (middle 80%: 56–77), with no trend over time (+0.1 years per decade, p = 0.81).
Development and growth laureates are the youngest (median 63) and economic history and labour the oldest (72, 70), but
the field differences are not significant (Kruskal–Wallis p = 0.60); laureates of three-way prizes are younger (median
64) than solo laureates (66.5) and those of two-way prizes (68). 48 of the 99 are alive (median age 79); the others died
at a median 85.9, 18 years after their prize. Data correction (10 October 2026): the API dates the 2022 prize
2011-10-10, so the ages of Bernanke, Diamond and Dybvig were 57, 57 and 56 in the crawl; section 1 recomputes them
from the prize year (68, 68, 67), which moved the finance median from 57 to 67 and the overall median from 66 to 67.

In [ ]:
recent = C3[C3.year >= 2016]
summary = {
    'prizes / laureates / works': f'{len(P)} / {len(L)} / {len(C3)}',
    'laureates per prize, 1969–79 vs 2020–25': f"{per.loc['1969–79', 'laureates per prize']} vs "
                                                 f"{per.loc['2020–25', 'laureates per prize']}",
    'share of laureates with a US affiliation': f"{geo.loc['all', 'share us']:.0%}",
    'models agree on the letter (all three)': f"{agreement.loc['letter', 'all three agree']:.0%}",
    'top letters (prizes)': ', '.join(f'{c} {v:.1f}' for c, v in F1['prizes'].head(5).items()),
    'level-2 fields awarded': len(F2),
    'letters never awarded': ', '.join(never.index),
    'last ten years (letters)': ', '.join(f'{y} {"+".join(sorted(seq[y]))}' for y in seq.index[-10:]),
    'repeats with the year before (observed vs random)': f"{rep.iloc[0]['observed repeats']} vs "
                                                          f"{rep.iloc[0]['expected (random order)']:.1f} "
                                                          f"(P = {rep.iloc[0]['P(random <= observed)']:.3f})",
    'women': f'{int(L.woman.sum())} of {len(L)} ({", ".join(map(str, W.year))})',
    'age at the award: median (middle 80%)': f'{a.median():.0f} ({q[0.10]:.0f}–{q[0.90]:.0f})',
    'age trend per decade of award': f'{fit.slope * 10:+.2f} years (p = {fit.pvalue:.2f})',
    'living laureates, median current age': f'{len(liv)}, {liv.age_now.median():.1f}'}
display(pd.Series(summary).to_frame('value'))
C3.drop(columns=['period']).to_csv(OUT / 'prize_works_jel.csv', index_label='work_id')
print('written:', ', '.join(sorted(p.name for p in OUT.iterdir())))